# -----------------------------------------------
# GPU Availability Check
# -----------------------------------------------
# First, we check if a GPU is available for PyTorch computations.
# This ensures that all subsequent operations can leverage CUDA acceleration.

In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
!nvidia-smi

Thu Apr  9 18:39:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


# -----------------------------------------------
# Setting Up the Device
# -----------------------------------------------
# We explicitly select the GPU as our computing device.
# If no GPU is found, you could fall back to CPU (not done here for brevity).

In [ ]:
# -----------------------------------------------
# Basic Matrix Multiplication
# -----------------------------------------------
# Create two random 1000x1000 matrices directly on the GPU.
# Perform standard matrix multiplication to demonstrate GPU usage.
# Preview the first 5x5 elements of the result.

In [4]:
import torch

# Set device to GPU
device = torch.device("cuda")

# Create two random matrices
x = torch.randn(1000, 1000, device=device)
y = torch.randn(1000, 1000, device=device)

# Matrix multiplication on GPU
z = torch.matmul(x, y)
print(z[:5, :5])  # preview first 5x5 elements

tensor([[  6.8606,  29.1020,  21.6460, -37.4627,  20.9714],
        [ -2.3855, -31.3760,  -2.3145, -19.7814, -49.6240],
        [ -1.1141,  -4.2101,  11.8112, -18.1986, -21.0868],
        [ 12.5552,  51.3516,   2.2858, -13.5199, -11.1958],
        [-14.6384,  24.7675,  44.1517,   6.7231, -18.5747]], device='cuda:0')


# -----------------------------------------------
# Now we demonstrate mixed precision using float16.
# Mixed precision leverages Tensor Cores for faster computations on modern NVIDIA GPUs.


In [5]:
# Create two large random matrices in float32
x = torch.randn(2000, 2000, device=device, dtype=torch.float32)
y = torch.randn(2000, 2000, device=device, dtype=torch.float32)

# Convert matrices to float16 (mixed precision)
x_half = x.half()
y_half = y.half()

# Perform matrix multiplication in mixed precision
torch.cuda.synchronize()  # make sure GPU is ready
import time
start = time.time()
z_half = torch.matmul(x_half, y_half)
torch.cuda.synchronize()  # wait for GPU to finish
elapsed_time = time.time() - start

print(f"Matrix multiplication in float16 completed in {elapsed_time:.4f} seconds")
print("Preview of result (first 5x5):")
print(z_half[:5, :5])

Matrix multiplication in float16 completed in 0.1750 seconds
Preview of result (first 5x5):
tensor([[ 36.5938, -25.6250, -68.3125,  19.3281,   6.8828],
        [ 12.4141,   9.9453, -38.7500,   2.0879,  24.8125],
        [ 41.0625, -27.7500,  -5.8047,  -2.3242, -10.8516],
        [  3.8379,  43.0000, -21.7656,  -2.0020, -28.1406],
        [ -4.8633, -27.0938,  -0.7632, -55.2500,  43.0312]], device='cuda:0',
       dtype=torch.float16)


In [6]:
# -----------------------------------------------
# Saving All Data for Showcase
# -----------------------------------------------
# Save matrices, results, and metrics so the project can be shared or reopened later.

import numpy as np
import pandas as pd
from google.colab import files

# 1️⃣ Save matrices as .pt (PyTorch format)
torch.save(x, "matrix_x.pt")
torch.save(y, "matrix_y.pt")
torch.save(z_half, "matrix_z_half.pt")

# 2️⃣ Save matrices as .npy (NumPy format, easy for preview)
np.save("matrix_x.npy", x.cpu().numpy())
np.save("matrix_y.npy", y.cpu().numpy())
np.save("matrix_z_half.npy", z_half.cpu().numpy())

# 3️⃣ Save performance metrics
metrics = {
    "Operation": ["Matrix Mult float16"],
    "Time_sec": [elapsed_time]
}
df_metrics = pd.DataFrame(metrics)
df_metrics.to_csv("gpu_performance.csv", index=False)

# 4️⃣ Optional: Save small preview (first 5x5 elements)
preview = z_half[:5, :5].cpu().numpy()
np.savetxt("preview_z_half.csv", preview, delimiter=",")

# 5️⃣ Zip everything for easy download
!zip -r gpu_project_results.zip *.pt *.npy *.csv

# 6️⃣ Download the zip
files.download("gpu_project_results.zip")

  adding: matrix_x.pt (deflated 7%)
  adding: matrix_y.pt (deflated 7%)
  adding: matrix_z_half.pt (deflated 8%)
  adding: matrix_x.npy (deflated 7%)
  adding: matrix_y.npy (deflated 7%)
  adding: matrix_z_half.npy (deflated 8%)
  adding: gpu_performance.csv (stored 0%)
  adding: preview_z_half.csv (deflated 73%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>